In [464]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [465]:
df = pd.read_csv('data/course_lead_scoring.csv')

In [466]:
df.head().T

,0,1,2,3,4
lead_source,paid_ads,social_media,events,paid_ads,referral
industry,NaN,retail,healthcare,retail,education
number_of_courses_viewed,1,1,5,2,3
annual_income,79450.0,46992.0,78796.0,83843.0,85012.0
employment_status,unemployed,employed,unemployed,NaN,self_employed
location,south_america,south_america,australia,australia,europe
interaction_count,4,1,3,1,3
lead_score,0.94,0.8,0.69,0.87,0.62
converted,1,0,1,0,1


In [467]:
df.isna().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [468]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [469]:
df.lead_source       = df.lead_source.fillna('NA')
df.industry          = df.industry.fillna('NA')
df.annual_income     = df.annual_income.fillna(0.0)
df.employment_status = df.employment_status.fillna('NA')
df.location          = df.location.fillna('NA')


In [470]:
df.isna().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [471]:
df['industry'].mode()[0]

'retail'

In [472]:
df['industry'].value_counts()


industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

#### Question 1
###### What is the most frequent observation (mode) for the column industry?
## Retail

# Creating the correlation matrix of numerical features

In [473]:
numerical_df = df.select_dtypes(include=['number'])

In [474]:
numerical_df

,number_of_courses_viewed,annual_income,interaction_count,lead_score,converted
0,1,79450.0,4,0.94,1
1,1,46992.0,1,0.80,0
2,5,78796.0,3,0.69,1
3,2,83843.0,1,0.87,0
4,3,85012.0,3,0.62,1
...,...,...,...,...,...
1457,1,0.0,4,0.53,1
1458,3,65259.0,2,0.24,1
1459,1,45688.0,3,0.02,1
1460,5,71016.0,0,0.25,1


In [475]:
corr_matrix = numerical_df.corr()

In [476]:
corr_matrix

,number_of_courses_viewed,annual_income,interaction_count,lead_score,converted
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879,0.435914
annual_income,0.009770,1.000000,0.027036,0.015610,0.053131
interaction_count,-0.023565,0.027036,1.000000,0.009888,0.374573
lead_score,-0.004879,0.015610,0.009888,1.000000,0.193673
converted,0.435914,0.053131,0.374573,0.193673,1.000000


### Question 2
##### Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

##### What are the two features that have the biggest correlation?

### Answer: annual_income and interaction_count

# Setting up the validation framework

In [477]:
from sklearn.model_selection import train_test_split

In [478]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [479]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

In [480]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [481]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

# Mutual Information

In [482]:
from sklearn.metrics import mutual_info_score

In [483]:
categorical_cols = df.select_dtypes(include=['object']).columns

In [484]:
categorical_cols

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object')

In [485]:
score = mutual_info_score(y_train, df_train.lead_source)
round(score, 2)

0.04

In [486]:
score = mutual_info_score(y_train, df_train.industry)
round(score, 2)


0.01

In [487]:
score = mutual_info_score(y_train, df_train.employment_status)
round(score, 2)

0.01

In [488]:
score = mutual_info_score(y_train, df_train.location)
round(score, 2)

0.0

### Question 3
##### Calculate the mutual information score between y and other categorical variables in the dataset. Use the training set only.
##### Round the scores to 2 decimals using round(score, 2)
##### Which of these variables has the biggest mutual information score?

### Answer: Lead source

# One-hot encoding

In [489]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score

In [490]:
numerical_cols = ['number_of_courses_viewed', 'annual_income', 'interaction_count',
        'lead_score']
categorical_cols

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object')

In [491]:
dv = DictVectorizer(sparse=False)


train_dict = df_train[list(categorical_cols) + numerical_cols].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[list(categorical_cols) + numerical_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [492]:
categorical_cols

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object')

# Logistic regression

In [493]:
from sklearn.linear_model import LogisticRegression

In [494]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [495]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [496]:
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
accuracy

0.6996587030716723

In [497]:
y_pred = model.predict_proba(X_val)[:, 1]

In [498]:
converted_decision = (y_pred >= 0.5)

In [499]:
(y_val == converted_decision).mean()

np.float64(0.6996587030716723)

In [500]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val

In [501]:
df_pred['correct'] = df_pred.prediction == df_pred.actual


In [502]:
round(df_pred.correct.mean(), 2)

np.float64(0.7)

### Question 4:
#### Answer is 0.7

In [503]:
df_pred

,probability,prediction,actual,correct
0,0.611922,1,0,False
1,0.799826,1,1,True
2,0.530213,1,0,False
3,0.471315,0,0,True
4,0.570661,1,0,False
...,...,...,...,...
288,0.419342,0,0,True
289,0.710539,1,1,True
290,0.418185,0,0,True
291,0.744835,1,1,True


# Feature Elimination Technique.

In [504]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'annual_income': np.float64(-0.0),
 'employment_status=NA': np.float64(-0.015),
 'employment_status=employed': np.float64(0.034),
 'employment_status=self_employed': np.float64(0.003),
 'employment_status=student': np.float64(0.012),
 'employment_status=unemployed': np.float64(-0.103),
 'industry=NA': np.float64(-0.025),
 'industry=education': np.float64(0.049),
 'industry=finance': np.float64(-0.02),
 'industry=healthcare': np.float64(-0.013),
 'industry=manufacturing': np.float64(-0.003),
 'industry=other': np.float64(-0.009),
 'industry=retail': np.float64(-0.032),
 'industry=technology': np.float64(-0.016),
 'interaction_count': np.float64(0.311),
 'lead_score': np.float64(0.051),
 'lead_source=NA': np.float64(0.02),
 'lead_source=events': np.float64(-0.012),
 'lead_source=organic_search': np.float64(-0.012),
 'lead_source=paid_ads': np.float64(-0.115),
 'lead_source=referral': np.float64(0.08),
 'lead_source=social_media': np.float64(-0.03),
 'location=NA': np.float64(0.004),
 'l

In [505]:
categorical_cols, numerical_cols

(Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object'),
 ['number_of_courses_viewed',
  'annual_income',
  'interaction_count',
  'lead_score'])

In [506]:
diff_list = []

In [507]:
small_feat = ['lead_source']

In [508]:
dv = DictVectorizer(sparse=False)


train_dict = df_train[small_feat].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[small_feat].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [509]:
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
small_accuracy = (y_val == converted_decision).mean()
diff = accuracy - small_accuracy
diff_list.append(float(diff))

In [510]:
accuracy, small_accuracy, diff, diff_list

(0.6996587030716723,
 np.float64(0.5631399317406144),
 np.float64(0.13651877133105794),
 [0.13651877133105794])

- lead_score -0.00682
- interaction_count  0.13993174061433444
- annual_income   -0.023890
- number_of_courses_viewed  0.09215017
- location 0.095563
- employment_status    0.129692
- industry   0.13651877

### Question 5: Answer is Lead_score

# Regularized Logistic Regression

In [511]:
dv = DictVectorizer(sparse=False)


train_dict = df_train[list(categorical_cols) + numerical_cols].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[list(categorical_cols) + numerical_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [512]:

C_grid = [0.01, 0.1, 1, 10, 100]
accuracies = []

for C in C_grid:

    # Create the model instance
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    print(C)
    converted_decision = (y_pred >= 0.5)
    accuracy = (y_val == converted_decision).mean()
    accuracies.append({C: float(round(accuracy, 3))})

print(accuracies)
    
    

0.01
0.1
1
10
100
[{0.01: 0.563}, {0.1: 0.563}, {1: 0.563}, {10: 0.563}, {100: 0.563}]


### Question 6: Answer is 0.01